In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
RUN_ID = 359           # change this to analyse a different run
ES_URL = "http://132.77.73.217:9200"
INDEX  = "event_log_v2"


In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from scipy.stats import gaussian_kde
from dateutil import parser as dateparser
from IPython.display import display


In [ ]:
def fetch_all_events(run_id):
    """Scroll through every event document for the given run_id."""
    scroll_url = f"{ES_URL}/{INDEX}/_search?scroll=2m"
    query = {
        "size": 1000,
        "query": {"term": {"run_id": run_id}},
        "sort": [{"timestamp": {"order": "asc"}}]
    }
    resp = requests.post(scroll_url, json=query,
                         headers={"Content-Type": "application/json"})
    resp.raise_for_status()
    data = resp.json()

    scroll_id = data.get("_scroll_id")
    hits = data["hits"]["hits"]
    total = data["hits"]["total"]
    total_val = total["value"] if isinstance(total, dict) else total
    print(f"Total matching events: {total_val}")

    all_docs = [h["_source"] for h in hits]

    while len(hits) == 1000:
        scroll_resp = requests.post(
            f"{ES_URL}/_search/scroll",
            json={"scroll": "2m", "scroll_id": scroll_id},
            headers={"Content-Type": "application/json"}
        )
        scroll_resp.raise_for_status()
        scroll_data = scroll_resp.json()
        hits = scroll_data["hits"]["hits"]
        all_docs.extend(h["_source"] for h in hits)
        scroll_id = scroll_data.get("_scroll_id", scroll_id)

    # clean up scroll context
    requests.delete(f"{ES_URL}/_search/scroll",
                    json={"scroll_id": scroll_id},
                    headers={"Content-Type": "application/json"})

    print(f"Fetched {len(all_docs)} documents for run_id={run_id}")
    return all_docs

docs = fetch_all_events(RUN_ID)


In [ ]:
def parse_ts(val):
    """Parse an ISO 8601 timestamp string to a datetime. Returns None on failure."""
    if not val:
        return None
    try:
        return dateparser.parse(str(val))
    except Exception:
        return None


events1 = []  # detected_light_on  — time marker: event_data.pi_timestamp
events2 = []  # state_transition_occurred — time marker: timestamp
events3 = []  # set_called               — time marker: timestamp
events4 = []  # detected_light_off — time marker: event_data.pi_timestamp

for doc in docs:
    # fields are nested under doc["event"]
    ev     = doc.get("event") or {}
    ed     = ev.get("event_data") or {}
    level  = ev.get("level")
    ts_top = parse_ts(doc.get("timestamp"))
    pi_ts  = parse_ts(ed.get("pi_timestamp"))

    # Event 1: id==LED2, level==1, pi_timestamp present
    if ed.get("id") == "LED2" and level == 1 and pi_ts is not None:
        events1.append(pi_ts)

    # Event 2: current_state == off_led2
    if ed.get("current_state") == "off_led2" and ts_top is not None:
        events2.append(ts_top)

    # Event 3: func_name==set, level==0, id==LED2
    if ed.get("func_name") == "set" and level == 0 and ed.get("id") == "LED2" and ts_top is not None:
        events3.append(ts_top)

    # Event 4: id==LED2, level==0, pi_timestamp present
    if ed.get("id") == "LED2" and level == 0 and pi_ts is not None:
        events4.append(pi_ts)

for lst in [events1, events2, events3, events4]:
    lst.sort()

print(f"Event counts:")
print(f"  E1 detected_light_on        : {len(events1)}")
print(f"  E2 state_transition_occurred: {len(events2)}")
print(f"  E3 set_called               : {len(events3)}")
print(f"  E4 detected_light_off       : {len(events4)}")


In [ ]:
def to_ms(dt_a, dt_b):
    return (dt_b - dt_a).total_seconds() * 1000.0


def pair_trials(e1, e2, e4):
    """
    Forward-nearest matching:
    For each e1[i], find nearest subsequent e2 > e1[i],
    then e4 > e2_found.
    """
    result = {"1_2": [], "2_4": [], "1_4": []}
    skipped = 0
    i2 = i4 = 0

    for t1 in e1:
        while i2 < len(e2) and e2[i2] <= t1:
            i2 += 1
        if i2 >= len(e2):
            skipped += 1
            continue
        t2 = e2[i2]

        while i4 < len(e4) and e4[i4] <= t2:
            i4 += 1
        if i4 >= len(e4):
            skipped += 1
            continue
        t4 = e4[i4]

        result["1_2"].append(to_ms(t1, t2))
        result["2_4"].append(to_ms(t2, t4))
        result["1_4"].append(to_ms(t1, t4))

    print(f"Paired trials: {len(result['1_4'])}   Skipped: {skipped}")
    return result


intervals = pair_trials(events1, events2, events4)

In [ ]:
INTERVAL_LABELS = {
    "1_2": "E1→E2  detected_light_on → state_transition",
    "2_4": "E2→E4  state_transition → set_called",
    "1_4": "E1→E4  end-to-end (light_on → light_off)",
}

rows = []
for key, label in INTERVAL_LABELS.items():
    arr = np.array(intervals[key])
    if len(arr) == 0:
        continue
    rows.append({
        "Interval": label,
        "N": len(arr),
        "Mean (ms)": round(arr.mean(), 3),
        "Median (ms)": round(np.median(arr), 3),
        "Std (ms)": round(arr.std(ddof=1), 3),
        "Variance (ms\u00b2)": round(arr.var(ddof=1), 3),
        "Min (ms)": round(arr.min(), 3),
        "Max (ms)": round(arr.max(), 3),
    })

stats_df = pd.DataFrame(rows).set_index("Interval")
pd.set_option("display.float_format", "{:.3f}".format)
display(stats_df)

In [ ]:
COLORS = ["#4C72B0", "#27AE60", "#E67E22"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Compute shared x and y limits across all intervals
all_arrs = [np.array(intervals[k]) for k in INTERVAL_LABELS if len(intervals[k]) >= 3]
x_min = min(a.min() for a in all_arrs)
x_max = max(a.max() for a in all_arrs)
x_pad = (x_max - x_min) * 0.05

# Pre-compute KDE peaks to set shared y limit
y_max = 0
for a in all_arrs:
    kde = gaussian_kde(a)
    xs  = np.linspace(x_min - x_pad, x_max + x_pad, 400)
    y_max = max(y_max, kde(xs).max())
y_max *= 1.15  # headroom

for idx, (key, label) in enumerate(INTERVAL_LABELS.items()):
    ax  = axes[idx]
    arr = np.array(intervals[key])
    col = COLORS[idx]

    if len(arr) < 3:
        ax.set_title(f"{label}\n(insufficient data, n={len(arr)})")
        continue

    ax.hist(arr, bins="auto", density=True, alpha=0.6, color=col, label="Histogram")

    kde = gaussian_kde(arr)
    xs  = np.linspace(x_min - x_pad, x_max + x_pad, 400)
    ax.plot(xs, kde(xs), color=col, linewidth=2.2, label="KDE")

    ax.axvline(arr.mean(), color="black", linestyle="--", linewidth=1.4,
               label=f"\u03bc = {arr.mean():.1f} ms")

    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_ylim(0, y_max)
    ax.set_xlabel("Interval (ms)", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.set_title(
        f"{label}\n"
        f"N={len(arr)},  \u03bc={arr.mean():.1f} ms,  \u03c3={arr.std(ddof=1):.1f} ms",
        fontsize=10
    )
    ax.legend(fontsize=8)

fig.suptitle(f"LED2 Interval Distributions — Run {RUN_ID}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("led2_interval_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor("#F8F9FA")

ax = fig.add_axes([0.0, 0.0, 1.0, 1.0])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_facecolor("#F8F9FA")

# ── Nodes (horizontal line) ───────────────────────────────────────────────
NX = [0.13, 0.50, 0.87]
NY = 0.52
HW = 0.105  # horizontal clearance to node box edge

NODE_LABELS = ["Detection\nof Event", "State\nTransition", "Action"]
NBOXES = [
    dict(boxstyle="round,pad=0.65", facecolor="#D6EAF8", edgecolor="#4C72B0", lw=2.5),
    dict(boxstyle="round,pad=0.65", facecolor="#D5F5E3", edgecolor="#27AE60", lw=2.5),
    dict(boxstyle="round,pad=0.65", facecolor="#FDEBD0", edgecolor="#E67E22", lw=2.5),
]
for x, lbl, nb in zip(NX, NODE_LABELS, NBOXES):
    ax.text(x, NY, lbl, ha="center", va="center", bbox=nb,
            fontsize=11, fontweight="bold", zorder=5,
            linespacing=1.4, color="#2C3E50")

# ── Straight arrows E1→E2 and E2→E4 ──────────────────────────────────────
for col, x0, x1 in [("#4C72B0", NX[0], NX[1]), ("#27AE60", NX[1], NX[2])]:
    ax.add_patch(FancyArrowPatch(
        posA=(x0+HW, NY), posB=(x1-HW, NY),
        connectionstyle="arc3,rad=0",
        arrowstyle="-|>", color=col, lw=2.5, mutation_scale=22,
        transform=ax.transData, zorder=4
    ))

# ── E1→E4 arc below the node line ─────────────────────────────────────────
# For a left→right arrow: rad > 0 bows the arc DOWNWARD.
# With rad=0.50, arc midpoint y ≈ NY - 0.5*rad*(NX[2]-HW - NX[0]-HW)
#   = 0.52 - 0.5*0.50*0.53 ≈ 0.387  — well below the straight arrows.
ax.add_patch(FancyArrowPatch(
    posA=(NX[0]+HW, NY), posB=(NX[2]-HW, NY),
    connectionstyle="arc3,rad=0.50",
    arrowstyle="-|>", color="#E67E22", lw=2.5, mutation_scale=22,
    transform=ax.transData, zorder=4
))

# ── Shared scale across all three distributions ───────────────────────────
INSET_KEYS = ["1_2", "2_4", "1_4"]
all_arrs_i = [np.array(intervals[k]) for k in INSET_KEYS if len(intervals[k]) >= 3]
xi_min = min(a.min() for a in all_arrs_i)
xi_max = max(a.max() for a in all_arrs_i)
xi_pad = (xi_max - xi_min) * 0.06
yi_max = 0
for a in all_arrs_i:
    kde = gaussian_kde(a)
    xs = np.linspace(xi_min - xi_pad, xi_max + xi_pad, 300)
    yi_max = max(yi_max, kde(xs).max())
yi_max *= 1.18

# ── Distribution insets ───────────────────────────────────────────────────
# All identical size (W×H).  Same x/y scale on every panel.
# E1→E2 and E2→E4 sit just above the node row (centred over each arrow).
# E1→E4 sits inside the downward arc (centred below the nodes).
W, H = 0.26, 0.20
mid12 = (NX[0] + NX[1]) / 2   # 0.315
mid24 = (NX[1] + NX[2]) / 2   # 0.685

INSET_SPECS = [
    ("1_2", "#4C72B0", "Detection → State Transition", mid12 - W/2, 0.67),
    ("2_4", "#27AE60", "State Transition → Action",    mid24 - W/2, 0.67),
    ("1_4", "#E67E22", "Detection → Action",           0.50  - W/2, 0.05),
]

for key, col, title, left, bot in INSET_SPECS:
    ax_i = fig.add_axes([left, bot, W, H])
    ax_i.set_facecolor("white")

    arr = np.array(intervals[key])
    if len(arr) >= 3:
        ax_i.hist(arr, bins="auto", density=True, alpha=0.50, color=col)
        kde = gaussian_kde(arr)
        xs  = np.linspace(xi_min - xi_pad, xi_max + xi_pad, 300)
        ax_i.plot(xs, kde(xs), color=col, lw=2)
        ax_i.axvline(arr.mean(), color="black", lw=1.3, linestyle="--",
                     label=f"\u03bc = {arr.mean():.1f} ms\n\u03c3 = {arr.std(ddof=1):.1f} ms")
        ax_i.legend(fontsize=7.5, frameon=True, framealpha=0.9,
                    edgecolor="#CCCCCC", loc="upper right")
    else:
        ax_i.text(0.5, 0.5, "no data", ha="center", va="center",
                  transform=ax_i.transAxes, fontsize=9, color="grey")

    ax_i.set_xlim(xi_min - xi_pad, xi_max + xi_pad)
    ax_i.set_ylim(0, yi_max)
    ax_i.set_xlabel("Interval (ms)", fontsize=8)
    ax_i.set_ylabel("Density", fontsize=8)
    ax_i.tick_params(labelsize=7)
    ax_i.set_title(title, fontsize=9, color=col, fontweight="bold", pad=4)
    ax_i.spines["top"].set_visible(False)
    ax_i.spines["right"].set_visible(False)
    for sp in ["left", "bottom"]:
        ax_i.spines[sp].set_linewidth(1.0)
        ax_i.spines[sp].set_edgecolor("#AAAAAA")

# Dotted connectors: distribution bottom → node top (makes association clear)
node_top = NY + 0.09   # approximate top of node boxes
inset_bot = 0.67
for x_mid, col in [(mid12, "#4C72B0"), (mid24, "#27AE60")]:
    ax.plot([x_mid, x_mid], [node_top, inset_bot],
            color=col, lw=1.2, linestyle=":", alpha=0.55, zorder=3)

fig.text(0.5, 0.975, f"LED2 Event Timing \u2014 Run {RUN_ID}",
         ha="center", va="top", fontsize=14, fontweight="bold", color="#2C3E50")

plt.savefig("led2_timing_graph.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()